# Video → npy 8576 chiều → train model bình luận (Colab, v3.9)

Luồng **từ đầu**, không dùng model đã train sẵn: video của bạn → npy → train model bình luận trên chính npy đó. Thêm video thì chỉ cần upload lên Drive rồi chạy lại: các trận đã trích được dùng lại, chỉ trích trận mới.

**Chạy lần lượt cell 1 → 13 từ trên xuống.** Nếu một cell báo lỗi thì dừng ở đó, đọc log ngay trong cell, sửa xong rồi mới chạy tiếp.

| Bước | Việc làm |
|---|---|
| 1–4 | GPU, cấu hình, mount Drive, lấy code + cài thư viện |
| 5 | Giải nén video, tải nhãn + npy Baidu gốc của đúng các trận có video |
| 6 | **Video → npy TimeSformer 768 chiều** (model Baidu fine-tune trên SoccerNet) |
| 7 | **Học phép ép 768 → 8576** trên các trận train. Bỏ qua nếu đã có trên Drive |
| 8 | **Tạo npy 8576 chiều từ video** cho mọi trận: `baidu_from_timesformer.npy` |
| 9 | Bảng độ giống với npy Baidu gốc |
| 10–11 | **Train model bình luận** (QFormer + GPT-2) **chỉ trên npy tạo từ video** |
| 12 | Chọn ngưỡng NMS + cách sinh câu trên valid, chấm trên test, xem câu bình luận mẫu |
| 13 | Timeline bình luận (sự kiện + thông tin ngoài lề) |

**Cần có trên Drive** (`MyDrive/soccer_project/`):
- Video trong `dataset/`, theo **một hoặc nhiều** cách sau:
  - các file `.zip` bất kỳ tên (ví dụ `Dataset.zip`, `Dataset_2.zip`...), bên trong có `<giải>/<mùa>/<trận>/` chứa video hiệp 1 và hiệp 2;
  - hoặc thư mục đã giải nén, cùng cấu trúc `<giải>/<mùa>/<trận>/`.
- `weights/*.pdparams`: weights TimeSformer.

**Tên file video:** bắt đầu bằng số hiệp `1` hoặc `2`, đuôi `.mkv`, `.mp4`, `.avi`, `.mov` hoặc `.webm`. Ví dụ: `1_224p.mkv`, `2_720p.mp4`, `1.mp4`. Độ phân giải nào cũng được, video được thu về 224 khi trích. Video 224p trích nhanh nhất; 720p giải mã chậm hơn vài lần. Thư mục trận phải đặt **đúng tên như SoccerNet** (ví dụ `england_epl/2014-2015/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley`), để khớp với nhãn.

**Cần bao nhiêu trận?** Với 10 trận (train 6), model bình luận bị "sập" về một câu lặp lại. Để model học được, nên có tối thiểu khoảng 30 trận, chia đủ cả 3 split train / valid / test theo danh sách chính thức của SoccerNet (cell 5 in ra trận nào thuộc split nào).

Chạy lại sau khi bị ngắt phiên: npy 768 chiều (`timesformer768/`) và phép ép (`sn-models/mapper_timesformer.pt`) đã lưu trên Drive sẽ được dùng lại, không trích lại video.

Chọn GPU: `Runtime → Change runtime type → T4 GPU`.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cấu hình

- `FEATURES`: file feature dùng để train model bình luận.
  - `"baidu_from_timesformer.npy"` (mặc định): npy 8576 chiều tạo từ video, cùng dạng với npy Baidu.
  - `"timesformer768.npy"`: dùng thẳng npy 768 chiều, không qua phép ép. Code tự nhận số chiều.
- `MODEL_NAME`: tên mới, để **không ghi đè** model `gpt2-30games` đã train trên npy Baidu thật.

In [ ]:
GITHUB_REPO = "https://github.com/ThuLinh3009/soccer_project.git"
GIT_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soccer_project"
WORK_DIR = "/content/soccernet-caption-deltalab"   # code
DATA = f"{WORK_DIR}/dataset/caption-2024"          # nhãn + npy
VIDEO_DIR = "/content/videos_raw"                  # video giải nén
TS_BATCH = 32                                      # T4: 32 (fp16). Hết VRAM thì giảm xuống 16

FEATURES = "baidu_from_timesformer.npy"            # hoặc "timesformer768.npy"
RETRAIN_MAPPER = False                             # True: học lại phép ép dù đã có trên Drive

MODEL_NAME = "gpt2-video10"                        # thư mục kết quả trong sn-models/
GPT_TYPE = "gpt2"
MAX_EPOCHS = 10
EVAL_FREQ = 5
BATCH_SIZE = 32
NUM_WORKERS = 2
NMS_THRESHOLD = 0.7

## 3. Mount Google Drive, tìm video và weights

In [ ]:
from google.colab import drive
import os, glob
drive.mount("/content/drive", force_remount=True)

if not os.path.isdir(DRIVE_ROOT):
    found = [p for p in glob.glob("/content/drive/MyDrive/**/soccer_project", recursive=True)
             + glob.glob("/content/drive/Shareddrives/*/soccer_project") if os.path.isdir(p)]
    assert found, f"Không thấy {DRIVE_ROOT}: thêm lối tắt tới thư mục soccer_project hoặc sửa DRIVE_ROOT ở cell 2"
    DRIVE_ROOT = found[0]

VIDEO_SRC = f"{DRIVE_ROOT}/dataset"                            # các file .zip và/hoặc thư mục video
TS_DRIVE_DIR = f"{DRIVE_ROOT}/timesformer768"                 # sao lưu npy 768 chiều
DRIVE_MODEL_DIR = f"{DRIVE_ROOT}/sn-models"                   # model bình luận + phép ép
MAPPER = f"{DRIVE_MODEL_DIR}/mapper_timesformer.pt"
weights = sorted(glob.glob(f"{DRIVE_ROOT}/**/*.pdparams", recursive=True))
TS_WEIGHTS = weights[0] if weights else None

print("DRIVE_ROOT :", DRIVE_ROOT)
ZIPS = sorted(glob.glob(f"{VIDEO_SRC}/**/*.zip", recursive=True))
import re
VIDEO_RE = re.compile(r"^([12])([_.\-][^/]*)?\.(mkv|mp4|avi|mov|webm)$", re.I)   # 1_224p.mkv, 2_720p.mp4, 1.mp4...
def is_video(path):
    return "__MACOSX" not in path and VIDEO_RE.match(os.path.basename(path)) is not None
MKVS = [p for p in glob.glob(f"{VIDEO_SRC}/**/*.*", recursive=True) if is_video(p)]
print("video      :", len(ZIPS), "file zip,", len(MKVS), "file video đã giải nén trong", VIDEO_SRC)
print("weights    :", TS_WEIGHTS or "❌ KHÔNG THẤY file .pdparams")
print("phép ép    :", MAPPER, "✅ (dùng lại)" if os.path.exists(MAPPER) else "(chưa có, cell 7 sẽ học)")
assert (ZIPS or MKVS) and TS_WEIGHTS

## 4. Lấy code từ GitHub và cài thư viện

- `nlg-eval` cài với `--no-deps`: các thư viện phụ thuộc cũ của nó không cài được trên Python mới, mà code chỉ cần BLEU / METEOR / ROUGE / CIDEr.
- METEOR cần Java. `JAVA_TOOL_OPTIONS` tắt các dòng cảnh báo `Cgroup` để METEOR đọc đúng điểm.

In [ ]:
if os.path.isdir(f"{WORK_DIR}/.git"):
    !git -C {WORK_DIR} fetch -q origin && git -C {WORK_DIR} reset -q --hard origin/{GIT_BRANCH}
else:
    !git clone -q -b {GIT_BRANCH} {GITHUB_REPO} {WORK_DIR}
%cd {WORK_DIR}
!git log --oneline -1
!grep -m1 "^## v" CHANGELOG.md
!pip install -q SoccerNet av tiktoken transformers wandb pycocoevalcap psutil six
!pip install -q --no-deps git+https://github.com/Maluuba/nlg-eval.git@master
!which java || (apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null)

import subprocess
os.environ["JAVA_TOOL_OPTIONS"] = "-XX:-UseContainerSupport -Xlog:disable"
os.environ["SN_LOCAL_SUBSET"] = "1"        # chỉ dùng các trận có file trên ổ
os.environ["SN_FEATURES"] = FEATURES       # bộ chấm điểm liệt kê trận theo file feature này
os.environ["WANDB_MODE"] = "offline"

def run(cmd):
    # chạy lệnh, in log trực tiếp; báo lỗi ngay tại cell này nếu lệnh thất bại
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=WORK_DIR)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): xem log ngay phía trên")

## 5. Giải nén video, tải nhãn và npy Baidu gốc

npy Baidu gốc chỉ dùng để **học phép ép** (cell 7) và **đo độ giống** (cell 9). Model bình luận ở cell 11 **không** dùng npy Baidu gốc.

In [ ]:
import zipfile
from huggingface_hub import hf_hub_download
from SoccerNet.utils import getListGames

# giải nén mọi file zip (chỉ các video chưa giải nén)
for zp in ZIPS:
    with zipfile.ZipFile(zp) as z:
        todo = [m for m in z.namelist() if is_video(m)
                and not os.path.exists(os.path.join(VIDEO_DIR, m))]
        if todo:
            print(f"giải nén {len(todo)} video từ {os.path.basename(zp)}")
            z.extractall(VIDEO_DIR, todo)

# gom video (mkv, mp4...) từ zip và từ thư mục trên Drive về một chỗ:
# VIDEO_ROOT/<giải>/<mùa>/<trận>/{1,2}_video (symlink). Một hiệp có nhiều độ phân giải thì ưu tiên 224p.
VIDEO_ROOT = "/content/videos_all"
found = sorted((p for root in (VIDEO_DIR, VIDEO_SRC) for p in glob.glob(f"{root}/**/*.*", recursive=True) if is_video(p)),
               key=lambda p: "224p" not in os.path.basename(p))
chosen = {}
for v in found:
    parts = v.split("/")
    key = (*parts[-4:-1], os.path.basename(v)[0])     # (<giải>, <mùa>, <trận>, hiệp)
    chosen.setdefault(key, v)
for (league, season, game, half), v in chosen.items():
    link = os.path.join(VIDEO_ROOT, league, season, game, f"{half}_video")
    if not os.path.lexists(link):
        os.makedirs(os.path.dirname(link), exist_ok=True)
        os.symlink(v, link)
VIDEO_GAMES = sorted({os.path.relpath(os.path.dirname(p), VIDEO_ROOT)
                      for p in glob.glob(f"{VIDEO_ROOT}/*/*/*/[12]_video")})
print(len(chosen), "video,", len(VIDEO_GAMES), "trận có video")
one_half = [g for g in VIDEO_GAMES if len(glob.glob(f"{VIDEO_ROOT}/{glob.escape(g)}/[12]_video")) < 2]
if one_half:
    print("⚠️ chỉ có 1 hiệp (bị bỏ qua khi train):", one_half)

for g in VIDEO_GAMES:
    for f, repo, rev in [("Labels-caption.json", "SoccerNet/SN-Labels", "main"),
                         ("1_baidu_soccer_embeddings.npy", "SoccerNet/SN-Features", "baidu-soccer-embeddings"),
                         ("2_baidu_soccer_embeddings.npy", "SoccerNet/SN-Features", "baidu-soccer-embeddings")]:
        if not os.path.exists(f"{DATA}/{g}/{f}"):
            hf_hub_download(repo, f"{g}/{f}", repo_type="dataset", revision=rev, local_dir=DATA)

split_of = {g: s for s in ["train", "valid", "test", "challenge"] for g in getListGames([s], task="caption")}
for g in VIDEO_GAMES:
    print(f"{split_of.get(g, '?'):6} {g}")
count = {s: sum(split_of.get(g) == s for g in VIDEO_GAMES) for s in ("train", "valid", "test")}
print("số trận theo split:", count)
assert all(count.values()), "Cần ít nhất 1 trận có video ở mỗi split train / valid / test"

## 6. Video → npy TimeSformer 768 chiều

- Mỗi hiệp cho ra `1_timesformer768.npy` / `2_timesformer768.npy`, mỗi giây một dòng.
- Các hiệp đã trích ở phiên trước được chép lại từ Drive (`timesformer768/`), chỉ trích phần còn thiếu.
- Trên T4: khoảng 5–10 phút mỗi hiệp nếu phải trích mới.

In [ ]:
import shutil
def sync(src_root, dst_root):
    for f in glob.glob(f"{src_root}/**/*_timesformer768.npy", recursive=True):
        dst = os.path.join(dst_root, os.path.relpath(f, src_root))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(f, dst)

if os.path.isdir(TS_DRIVE_DIR):
    sync(TS_DRIVE_DIR, DATA)
try:
    run(["python", "timesformer_extract.py", "--weights", TS_WEIGHTS, "--video_root", VIDEO_ROOT, "--video_name", "{half}_video",
         "--out_root", DATA, "--batch_size", str(TS_BATCH)])
finally:
    sync(DATA, TS_DRIVE_DIR)   # lưu cả các hiệp đã xong nếu bị lỗi giữa chừng

missing = [f"{g}/{h}" for g in VIDEO_GAMES for h in (1, 2)
           if not os.path.exists(f"{DATA}/{g}/{h}_timesformer768.npy")]
print(2 * len(VIDEO_GAMES) - len(missing), "/", 2 * len(VIDEO_GAMES), "hiệp đã có npy 768 chiều")
assert not missing, f"Còn thiếu: {missing}. Chạy lại cell này"

## 7. Học phép ép 768 → 8576

| Nhóm trận | Dùng để |
|---|---|
| train | Học phép ép (ridge regression, 5 giây TimeSformer quanh giây t, tự tìm độ lệch thời gian) |
| valid | Chọn tham số |
| test | Đo độ giống. Không dùng để học |

Nếu `mapper_timesformer.pt` đã có trên Drive (lần chạy trước) thì bỏ qua, trừ khi `RETRAIN_MAPPER = True`.

In [ ]:
if FEATURES == "timesformer768.npy":
    print("FEATURES = timesformer768.npy: không cần phép ép, bỏ qua cell 7–9")
elif os.path.exists(MAPPER) and not RETRAIN_MAPPER:
    print("Dùng lại phép ép đã học:", MAPPER)
else:
    run(["python", "feature_mapper.py", "train", "--soccernet_path", DATA, "--out", MAPPER,
         "--train_splits", "train", "--val_splits", "valid", "--test_splits", "test"])

## 8. Tạo npy 8576 chiều từ video cho mọi trận

Áp phép ép lên npy 768 chiều của **cả train, valid và test**. Mỗi hiệp được `1_/2_baidu_from_timesformer.npy`, kích thước (số giây, 8576), cùng dạng npy Baidu.
Đây chính là feature mà model bình luận sẽ học ở cell 11.

In [ ]:
if FEATURES != "timesformer768.npy":
    run(["python", "feature_mapper.py", "apply", "--soccernet_path", DATA, "--mapper", MAPPER,
         "--splits", "train", "valid", "test", "--out_name", FEATURES, "--overwrite"])

import numpy as np
for g in VIDEO_GAMES:
    shapes = [np.load(f"{DATA}/{g}/{h}_{FEATURES}", mmap_mode="r").shape for h in (1, 2)]
    print(f"{split_of.get(g, '?'):6} {shapes}  {g}")

## 9. Độ giống với npy Baidu gốc (để tham khảo)

`cosine`: độ giống trung bình (1 = giống hệt). Mốc: hai giây liền nhau của Baidu khoảng 0,94, hai giây ngẫu nhiên khoảng 0,67.

In [ ]:
import json, pandas as pd
REPORT = MAPPER.replace(".pt", "_report.json")
if FEATURES != "timesformer768.npy" and os.path.exists(REPORT):
    rep = json.load(open(REPORT))
    print("offset:", rep["offset"], "| context:", rep["context"], "| lambda:", rep["lambda"], "| kind:", rep["kind"])
    display(pd.DataFrame({k: rep[k] for k in ("val", "test") if k in rep}).round(4))
    display(pd.DataFrame({k.replace("reference_", "mốc_"): rep[k] for k in ("reference_val", "reference_test") if k in rep}).round(4))

## 10. Lưu `models/` thẳng lên Drive

Symlink `models → sn-models` trên Drive, để checkpoint không mất khi hết phiên Colab. Model mới nằm ở `sn-models/{MODEL_NAME}`, không đụng tới `gpt2-30games`.

In [ ]:
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
if not os.path.islink("models"):
    !rm -rf models
!ln -sfn "{DRIVE_MODEL_DIR}" models
!ls models

## 11. Train model bình luận trên npy tạo từ video

`main.py` chạy lần lượt: **classifying** (phân loại sự kiện), **captioning** (GPT-2 sinh lời bình), **spotting** (tìm thời điểm sự kiện), **dvc** (ghép lại và chấm điểm trên test).

- Chỉ dùng các trận có `FEATURES` + nhãn trên ổ, tức **các trận có video**: train 6, valid 2, test 2.
- Model bình luận **không thấy** npy Baidu gốc. Mọi feature đều đi ra từ video.
- Số trận rất ít so với 30 trận của `gpt2-30games`, nên điểm sẽ thấp hơn. Muốn điểm tốt hơn thì cần thêm video.

In [ ]:
run(["python", "main.py", "--SoccerNet_path", DATA, "--features", FEATURES,
     "--model_name", MODEL_NAME, "--GPU", "0", "--pool", "QFormer", "--model_type", "gpt", "--gpt_type", GPT_TYPE,
     "--NMS_threshold", str(NMS_THRESHOLD), "--split_test", "test", "--max_epochs", str(MAX_EPOCHS),
     "--evaluation_frequency", str(EVAL_FREQ), "--batch_size", str(BATCH_SIZE), "--max_num_worker", str(NUM_WORKERS)])

## 12. Chọn cách sinh câu trên valid, chấm trên test

- `NMS_threshold` thấp hơn: báo nhiều sự kiện hơn, ít bỏ sót hơn nhưng dễ báo sai.
- `top_k` = 1 luôn chọn từ chắc nhất nên dễ lặp câu; `top_k` = 3 đa dạng hơn.

Thử các tổ hợp trên **valid**, lấy tổ hợp có `METEOR_loose` cao nhất rồi chấm trên **test**. Không train lại.

In [ ]:
import pandas as pd
log = sorted(glob.glob(f"models/{MODEL_NAME}/*.log"))[-1]
print("Log train:", log)
!grep -E "METEOR|Bleu_4|CIDEr|a_mAP" "{log}" | tail -15

common = ["--SoccerNet_path", DATA, "--model_name", MODEL_NAME, "--features", FEATURES, "--gpt_type", GPT_TYPE]
run(["python", "retune.py", *common, "--split", "valid", "--nms_thresholds", "0.3", "0.4", "0.5", "0.7", "--top_k", "1", "3"])
val = pd.read_csv(f"models/{MODEL_NAME}/retune_valid.csv")
best = val.loc[val["METEOR_loose"].idxmax()]
BEST_NMS, BEST_TOPK = float(best["nms_threshold"]), int(best["top_k"])
print("tốt nhất trên valid: NMS", BEST_NMS, "top_k", BEST_TOPK)
run(["python", "retune.py", *common, "--split", "test", "--nms_thresholds", str(BEST_NMS), "--top_k", str(BEST_TOPK)])
display(pd.read_csv(f"models/{MODEL_NAME}/retune_test.csv").T)

files = sorted(glob.glob(f"models/{MODEL_NAME}/outputs/test/**/results_dense_captioning.json", recursive=True))
if files:
    print("\n", files[0])
    for p in json.load(open(files[0]))["predictions"][:15]:
        print(p.get("gameTime"), "|", p.get("label"), "|", p.get("comment"))

## 13. Timeline bình luận

Ghép lời bình sự kiện của model (luồng YES) với thông tin ngoài lề ở những khoảng im lặng (luồng NO): đội hình, tỉ số, bảng xếp hạng...

In [ ]:
run(["python", "commentary_timeline.py", "--soccernet_path", DATA, "--predictions", f"models/{MODEL_NAME}/outputs/test"])
tl = sorted(glob.glob(f"models/{MODEL_NAME}/outputs/test/**/commentary_timeline.txt", recursive=True))
print(tl[0])
!head -60 "{tl[0]}"